In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from groq import Groq

In [3]:
client = Groq()

In [4]:
models = client.models.list()

In [5]:
model_ids = []
for model in models.data:
    model_ids.append(model.id)

In [6]:
model_ids

['allam-2-7b',
 'meta-llama/llama-4-scout-17b-16e-instruct',
 'canopylabs/orpheus-arabic-saudi',
 'whisper-large-v3',
 'qwen/qwen3-32b',
 'llama-3.3-70b-versatile',
 'whisper-large-v3-turbo',
 'meta-llama/llama-prompt-guard-2-22m',
 'openai/gpt-oss-120b',
 'canopylabs/orpheus-v1-english',
 'llama-3.1-8b-instant',
 'openai/gpt-oss-safeguard-20b',
 'groq/compound',
 'openai/gpt-oss-20b',
 'meta-llama/llama-prompt-guard-2-86m',
 'groq/compound-mini']

### Text Generation

In [10]:
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.3-70b-versatile"
)
print(chat_completion.choices[0].message.content)

Fast language models are crucial in natural language processing (NLP) and have numerous benefits. Here are some reasons why they're important:

1. **Improved User Experience**: Fast language models enable applications to respond quickly to user queries, making the interaction more engaging and efficient. This is particularly important for real-time applications, such as chatbots, virtual assistants, and language translation software.
2. **Increased Productivity**: By reducing the time it takes to process and generate text, fast language models can significantly boost productivity in various industries, including customer service, content creation, and data analysis.
3. **Enhanced Scalability**: Fast language models can handle large volumes of data and user requests, making them ideal for applications with high traffic or large user bases. This scalability is essential for businesses and organizations that need to process vast amounts of text data.
4. **Better Performance in Real-World 

### Reasoning

In [11]:
completion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "How many r's are in the word strawberry?"
        }
    ],
    temperature=0.6,
    max_completion_tokens=1024,
    top_p=0.95,
)

In [13]:
completion.choices[0].message.content

'There are **two** “r” letters in the word *strawberry*.'

### Structured Outputs

In [17]:
import json

In [15]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": "Extract product review information from the text."},
        {
            "role": "user",
            "content": "I bought the UltraSound Headphones last week and I'm really impressed! The noise cancellation is amazing and the battery lasts all day. Sound quality is crisp and clear. I'd give it 4.5 out of 5 stars.",
        },
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "product_review",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "product_name": {"type": "string"},
                    "rating": {"type": "number"},
                    "sentiment": {
                        "type": "string",
                        "enum": ["positive", "negative", "neutral"]
                    },
                    "key_features": {
                        "type": "array",
                        "items": {"type": "string"}
                    }
                },
                "required": ["product_name", "rating", "sentiment", "key_features"],
                "additionalProperties": False
            }
        }
    }
)

In [18]:
result = json.loads(response.choices[0].message.content or "{}")

In [19]:
print(json.dumps(result, indent=2))

{
  "product_name": "UltraSound Headphones",
  "rating": 4.5,
  "sentiment": "positive",
  "key_features": [
    "noise cancellation",
    "battery lasts all day",
    "sound quality"
  ]
}


### Caching

In [42]:
long_system = """
You are a helpful assistant.
""" * 500

initial_messages = [
    {
        "role": "system",
        "content": long_system
    },
    {
        "role": "user",
        "content": "What is quantum computing?"
    }
]

# First request - creates cache for system message
first_response = client.chat.completions.create(
    messages=initial_messages,
    model="openai/gpt-oss-120b",
    max_completion_tokens=200,
)

print("First response:", first_response.choices[0].message.content)

First response: **Quantum computing** is a type of computation that uses the principles of quantum mechanics—the physics that governs the behavior of particles at the atomic and sub‑atomic scales—to process information.

Below is a high‑level overview of how it differs from classical (ordinary) computing and what makes it powerful.

---

## 1. Classical vs. Quantum Bits  

| Classical Computing | Quantum Computing |
|---------------------|-------------------|
| **Bit** – the basic unit of information. It can be either **0** *or* **1**. | **Qubit** – the basic unit of information. It can be **0**, **1**, **or**


In [43]:
print("Usage:", first_response.usage)

Usage: CompletionUsage(completion_tokens=200, prompt_tokens=3079, total_tokens=3279, completion_time=0.434423884, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=59), prompt_time=0.123272705, prompt_tokens_details=None, queue_time=0.053362394, total_time=0.557696589)


In [47]:
conversation_messages = [
    *initial_messages,
    # first_response.choices[0].message,
    # {
    #     "role": "user",
    #     "content": "Can you give me a simple example of how quantum superposition works?"
    # }
]

second_response = client.chat.completions.create(
    messages=conversation_messages,
    model="openai/gpt-oss-120b"
)

print("Second response:", second_response.choices[0].message.content)

Second response: **Quantum computing** is a model of computation that uses the principles of quantum mechanics to process information. Unlike classical computers, which encode data in bits that are either 0 or 1, quantum computers use **quantum bits** or **qubits**, which can exist in a superposition of both 0 and 1 simultaneously. This property, together with two other uniquely quantum phenomena—**entanglement** and **interference**—gives quantum computers the potential to solve certain types of problems much more efficiently than classical machines.

### Core Concepts

| Concept | Classical Analog | Quantum Version | Why It Matters |
|---------|------------------|----------------|----------------|
| **Bit** | 0 or 1 | **Qubit**: | A qubit can be in a linear combination α|0⟩ + β|1⟩, where |α|² + |β|² = 1. |
| **Superposition** | Not possible (a bit is either 0 or 1) | A qubit can simultaneously represent both 0 and 1. | Enables parallelism: a register of *n* qubits can represent 2ⁿ st

In [48]:
print("Usage:", second_response.usage)

Usage: CompletionUsage(completion_tokens=1205, prompt_tokens=3079, total_tokens=4284, completion_time=2.539341196, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=29), prompt_time=0.156798468, prompt_tokens_details=None, queue_time=0.060911651, total_time=2.696139664)


In [50]:
second_response

ChatCompletion(id='chatcmpl-1c304cb8-8883-4ffd-b7d4-f876e3a842a5', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='**Quantum computing** is a model of computation that uses the principles of quantum mechanics to process information. Unlike classical computers, which encode data in bits that are either\u202f0\u202for\u202f1, quantum computers use **quantum bits** or **qubits**, which can exist in a superposition of both 0 and 1 simultaneously. This property, together with two other uniquely quantum phenomena—**entanglement** and **interference**—gives quantum computers the potential to solve certain types of problems much more efficiently than classical machines.\n\n### Core Concepts\n\n| Concept | Classical Analog | Quantum Version | Why It Matters |\n|---------|------------------|----------------|----------------|\n| **Bit** | 0 or 1 | **Qubit**: | A qubit can be in a linear combination\u202fα|0⟩\u202f+\u202fβ|1⟩, where |α|²\u202f+\

### Tool Use

In [53]:
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user", 
            "content": "What happened in AI last week? Give me a concise, one paragraph summary of the most important events."
        }
    ],
    model="openai/gpt-oss-20b",
    temperature=1,
    max_completion_tokens=2048,
    top_p=1,
    stream=False,
    stop=None,
    tool_choice="required",
    tools=[
        {
            "type": "browser_search"
        }
    ]
)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01k5jpzea1erpaecyhrx62dpcs` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7878, Requested 2349. Please try again in 16.7025s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
print(chat_completion.choices[0].message.content)

### Remote Tools

In [8]:
import openai
import os

client = openai.OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [9]:
response = client.responses.create(
    model="openai/gpt-oss-120b",
    input="What models are trending on Huggingface?",
    tools=[
        {
            "type": "mcp",
            "server_label": "Huggingface",
            "server_url": "https://huggingface.co/mcp",
        }
    ]
)

BadRequestError: Error code: 400 - {'error': {'message': 'Tool call validation failed: tool call validation failed: parameters for tool hub_repo_search did not match schema: errors: [`/author`: expected string, but got null, `/filters`: expected array, but got null]', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "hub_repo_search", "arguments": {\n  "author": null,\n  "filters": null,\n  "limit": 10,\n  "query": "",\n  "repo_types": ["model"],\n  "sort": "trendingScore"\n}}'}}

### Local Tool Calling

In [4]:
client = Groq()

In [6]:
calculate_tool_schema = {
  "type": "function",
  "function": {
    "name": "calculate",
    "description": "Evaluate a mathematical expression",
    "parameters": {
      "type": "object",
      "properties": {
        "expression": {
          "type": "string",
          "description": "The mathematical expression to evaluate"
        }
      },
      "required": ["expression"]
    }
  }
}

In [22]:
import json

def calculate(expression: str) -> str:
    """Execute the calculation"""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

available_functions = {
    "calculate": calculate,
}

def execute_tool_call(tool_call):
    """Parse and execute a single tool call"""
    function_name = tool_call.function.name
    function_to_call = available_functions[function_name]
    function_args = json.loads(tool_call.function.arguments)
    
    # Call the function with unpacked arguments
    return function_to_call(**function_args)

In [23]:
# 1. Call model with tool schema
messages = [{"role": "user", "content": "What is 25 * 4?"}]

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages,
    tools=[calculate_tool_schema]
)

In [9]:
response.choices[0].message

ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='User asks a simple multiplication. Use calculate function.', tool_calls=[ChatCompletionMessageToolCall(id='fc_8e7d3df4-340c-42f7-b227-6fc1ff31b317', function=Function(arguments='{"expression":"25 * 4"}', name='calculate'), type='function')])

In [10]:
messages.append(response.choices[0].message)

In [24]:
if response.choices[0].message.tool_calls:
    for tool_call in response.choices[0].message.tool_calls:
        function_response = execute_tool_call(tool_call)
        
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "name": tool_call.function.name,
            "content": str(function_response)
        })        

In [14]:
messages

[{'role': 'user', 'content': 'What is 25 * 4?'},
 ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='User asks a simple multiplication. Use calculate function.', tool_calls=[ChatCompletionMessageToolCall(id='fc_8e7d3df4-340c-42f7-b227-6fc1ff31b317', function=Function(arguments='{"expression":"25 * 4"}', name='calculate'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'fc_8e7d3df4-340c-42f7-b227-6fc1ff31b317',
  'name': 'calculate',
  'content': '100'}]

In [15]:
final = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages
)

In [21]:
final.choices[0].message.content

'The result of \\(25 \\times 4\\) is **100**.'